# Phase 3 &mdash; Bronze Layer

This is the first stage of our Medallion pipeline. Its job is to take the Lending Club loan data exactly as it arrived and register it as a managed Delta table. No cleaning, no filtering, no type changes &mdash; that all happens in the Silver layer. Keeping Bronze untouched means we always have a faithful record of the source, and that any bug we introduce later can be traced back by re-reading the raw table.

### Input
The 2014&ndash;2017 Lending Club CSV produced by `src/01_data_loading.py`. We uploaded it through the Databricks Catalog UI, which automatically converted the upload into a managed Delta table at `workspace.default.optimized_data_14_17`.

### Output
`workspace.default.bronze_loans` &mdash; an identical copy of the uploaded table, renamed under our medallion convention.

### Why a sanitization step?
The pandas export that produced the CSV left an `Unnamed: 0` index column. Delta rejects column names containing spaces, colons, or other punctuation, so we run a one-pass rename that replaces any non-alphanumeric character with an underscore and lowercases every name. It is a *cosmetic* change &mdash; the row values are identical to the uploaded data &mdash; but it lets Delta accept the schema.

In [0]:
# A SparkSession named `spark` is already available on a Databricks cluster.
# The try/except keeps the notebook runnable outside Databricks too (e.g. in a
# local `pip install pyspark delta-spark` environment for development).
try:
    spark  # type: ignore[name-defined]
except NameError:
    from pyspark.sql import SparkSession
    spark = (
        SparkSession.builder.appName("phase3-bronze")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .getOrCreate()
    )

print(f"Spark {spark.version} ready.")

Spark 4.1.0 ready.


In [0]:
# Source and destination tables. Update SOURCE_TABLE if the uploaded CSV landed
# somewhere other than workspace.default.
SOURCE_PATH = "/Volumes/workspace/default/raw_data/optimized_data_14_17.csv"
BRONZE_TABLE = "workspace.default.bronze_loans"

## Step 1 &mdash; Load the uploaded table

No transformations. Just read it into a Spark DataFrame and confirm the shape.

In [0]:
raw_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(SOURCE_PATH)
)

print(f"Rows: {raw_df.count():,}")
print(f"Columns: {len(raw_df.columns)}")
display(raw_df.limit(5))

Rows:    1,534,710
Columns: 142


## Step 2 &mdash; Sanity-check with a few rows and the schema

In [0]:
raw_df.limit(5).toPandas()

,Unnamed: 0,id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,url,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,...,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_fico_range_low,sec_app_fico_range_high,sec_app_earliest_cr_line,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,sec_app_chargeoff_within_12_mths,sec_app_collections_12_mths_ex_med,hardship_flag,hardship_type,hardship_reason,hardship_status,deferral_term,hardship_amount,hardship_start_date,hardship_end_date,payment_plan_start_date,hardship_length,hardship_dpd,hardship_loan_status,orig_projected_additional_accrued_interest,hardship_payoff_balance_amount,hardship_last_payment_amount,debt_settlement_flag
0,80428,63186613,16550,16550,16550.0,60 months,13.33%,379.37,C,C3,Ground Maintenance,< 1 year,MORTGAGE,36000.0,Verified,2015-11-01,Charged Off,n,https://lendingclub.com/browse/loanDetail.acti...,debt_consolidation,Debt consolidation,782xx,TX,29.17,0,Jun-2000,685,689,1.0,52.0,NaN,12,0,18346,53.8%,24,f,0.00,0.00,8301.04,...,0.0,0.0,0.0,2.0,91.7,25.0,0,0,95581.0,37695.0,21800.0,23597.0,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,None,None,None,NaN,NaN,None,None,None,NaN,NaN,None,NaN,NaN,NaN,N
1,80429,63216902,18000,18000,18000.0,60 months,12.29%,403.05,C,C1,nurse,10+ years,OWN,95000.0,Not Verified,2015-11-01,Charged Off,n,https://lendingclub.com/browse/loanDetail.acti...,major_purchase,Major purchase,701xx,LA,10.26,1,Oct-1998,740,744,1.0,9.0,NaN,13,0,13908,12.1%,22,f,0.00,0.00,14094.46,...,0.0,0.0,0.0,2.0,90.9,11.1,0,0,350500.0,38908.0,111700.0,25000.0,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,None,None,None,NaN,NaN,None,None,None,NaN,NaN,None,NaN,NaN,NaN,N
2,80431,63220466,15000,15000,15000.0,60 months,15.61%,361.67,D,D1,Senior Logistics Support Specialist,< 1 year,RENT,52000.0,Not Verified,2015-11-01,Charged Off,n,https://lendingclub.com/browse/loanDetail.acti...,small_business,Business,921xx,CA,26.31,0,Feb-2004,745,749,0.0,NaN,NaN,6,0,9806,29.8%,30,w,0.00,0.00,11461.99,...,0.0,0.0,0.0,1.0,100.0,0.0,0,0,69630.0,34654.0,13000.0,36730.0,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,INTEREST ONLY-3 MONTHS DEFERRAL,UNEMPLOYMENT,BROKEN,3.0,116.25,Jun-2018,Jul-2018,Jul-2018,3.0,5.0,Current,NaN,9071.66,171.38,N
3,80432,63290549,20000,20000,19900.0,60 months,14.65%,472.14,C,C5,Project Accountant,< 1 year,MORTGAGE,60000.0,Verified,2015-11-01,Charged Off,n,https://lendingclub.com/browse/loanDetail.acti...,debt_consolidation,Debt consolidation,328xx,FL,29.68,0,Oct-1998,710,714,1.0,NaN,89.0,11,1,8720,32%,19,w,0.00,0.00,17191.80,...,NaN,0.0,0.0,4.0,100.0,20.0,1,0,88180.0,57593.0,13050.0,61330.0,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,None,None,None,NaN,NaN,None,None,None,NaN,NaN,None,NaN,NaN,NaN,Y
4,80434,63349632,19600,19600,19600.0,60 months,17.86%,496.22,D,D5,Asset and Profit Protection Manager,2 years,RENT,49000.0,Source Verified,2015-11-01,Current,n,https://lendingclub.com/browse/loanDetail.acti...,debt_consolidation,Debt consolidation,958xx,CA,31.57,0,Sep-1995,685,689,0.0,57.0,NaN,6,0,42318,99.7%,13,w,2828.17,2828.17,26902.83,...,0.0,0.0,0.0,1.0,84.6,100.0,0,0,107738.0,63436.0,15600.0,26054.0,NaN,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,None,None,None,NaN,NaN,None,None,None,NaN,NaN,None,NaN,NaN,NaN,N


In [0]:
raw_df.printSchema()

root
 |-- Unnamed: 0: long (nullable = true)
 |-- id: long (nullable = true)
 |-- loan_amnt: long (nullable = true)
 |-- funded_amnt: long (nullable = true)
 |-- funded_amnt_inv: double (nullable = true)
 |-- term: string (nullable = true)
 |-- int_rate: string (nullable = true)
 |-- installment: double (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- emp_title: string (nullable = true)
 |-- emp_length: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_inc: double (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- issue_d: date (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- pymnt_plan: string (nullable = true)
 |-- url: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- title: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- addr_state: string (nullable = true)
 |-- dti: double (nullable = true)
 |-- delinq_2yrs: long (nul

## Step 3 &mdash; Sanitize column names

Replace any character outside `[A-Za-z0-9_]` with `_` and lowercase the result. In our data this only renames `Unnamed: 0` (a pandas index artifact) to `unnamed_0`, but the same rule safely handles anything else.

In [0]:
import re

def sanitize(name):
    cleaned = re.sub(r"[^A-Za-z0-9_]+", "_", name).strip("_").lower()
    return cleaned or "col"

renamed_df = raw_df
changed = []
for old in raw_df.columns:
    new = sanitize(old)
    if new != old:
        renamed_df = renamed_df.withColumnRenamed(old, new)
        changed.append((old, new))

print(f"Renamed {len(changed)} column(s):")
for old, new in changed:
    print(f"  {old!r:<20} -> {new!r}")

Renamed 1 column(s):
  'Unnamed: 0'         -> 'unnamed_0'


## Step 4 &mdash; Write the Bronze Delta table

`mode("overwrite")` makes re-runs idempotent &mdash; the table ends in the same state whether this is the first run or the tenth.

In [0]:
(
    renamed_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

print(f"Wrote Delta table: {BRONZE_TABLE}")

Wrote Delta table: workspace.default.bronze_loans


## Step 5 &mdash; Verify

In [0]:
spark.sql(f"SELECT COUNT(*) AS row_count FROM {BRONZE_TABLE}").show()
spark.sql(f"DESCRIBE DETAIL {BRONZE_TABLE}") \
    .select("name", "location", "numFiles", "sizeInBytes") \
    .show(truncate=False)

+---------+
|row_count|
+---------+
|  1534710|
+---------+

+------------------------------+--------+--------+-----------+
|name                          |location|numFiles|sizeInBytes|
+------------------------------+--------+--------+-----------+
|workspace.default.bronze_loans|        |4       |176287407  |
+------------------------------+--------+--------+-----------+

